# Speeding up training
You have probably noticed how slow training can get when running for many timesteps. This notebook explains how to transition from single-step to multi-step mode and how this can help you make the most of your hardware. In some cases, we have achieved substantial speedups, though the actual gains will depend on how much additional parallelization your setup can leverage.

Multi-step mode is different from single-step mode by how it passes tensors around and how the computations are done. Where single-step mode takes input for a single timestep, multi-step mode takes input from many timesteps. Computations are applied to all these timesteps at once, if possible.

## The data, network, standard setup
This you should all know by now. We prepare a fully-connected network for SHD. Notice one important change, we now take `step_mode` as argument for the `forward` and pass it to all neuron layers, and flatten differently if in multi-step mode. This is necessary to let all components know what mode we are using to apply the transformation in the right way.

In [1]:
import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron, SpikingGammaOutput
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import compute_linspace_bucket_decay_rates, SpikeConfig
from spikinggamma.utils.enums import StepMode
from spikinggamma.utils.helpers import MultiStepCrossEntropyLoss, multi_split_step_forward
from load_data import get_shd_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Data
train_dataset, test_dataset, input_shape, n_outputs, T = get_shd_dataset(timestep_size=3600)

# Hyperparameters
n_buckets = 10
decay_factor = 0.1
spike_config = SpikeConfig(
    spike_threshold=0.3
)
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_buckets, decay_factor)

# A simple fully connected network with 3 hidden layers
class SpikingGammaNet(nn.Module):
    def __init__(self, 
                 input_shape, 
                 n_outputs,
                 n_buckets,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()
        layer1_shape_out = (256,)
        layer2_shape_out = (256,)
        layer3_shape_out = (256,)
        layer4_shape_out = (n_outputs,)

        self.hid1 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer1_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(input_shape[0], layer1_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(layer1_shape_out))
        )

        self.hid2 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer2_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(layer1_shape_out[0], layer2_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(layer2_shape_out))
        )

        self.hid3 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer3_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(layer2_shape_out[0], layer3_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(layer3_shape_out))
        )

        self.out = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer4_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(layer3_shape_out[0], layer4_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(layer4_shape_out))
        )

        self.spike_out = SpikingGammaOutput(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config)
        )

    def reset(self):
        self.hid1.reset()
        self.hid2.reset()
        self.hid3.reset()
        self.out.reset()
        self.spike_out.reset()

    def detach(self):
        self.hid1.detach()
        self.hid2.detach()
        self.hid3.detach()
        self.out.detach()
        self.spike_out.detach()

    def forward(self, s_in, step_mode: StepMode):
        s_in = s_in.flatten(start_dim=1 if step_mode == StepMode.SINGLE else 2)
        y_hid1 = self.hid1(s_in, step_mode)
        y_hid2 = self.hid2(y_hid1, step_mode)
        y_hid3 = self.hid3(y_hid2, step_mode)
        y_out = self.out(y_hid3, step_mode)
        y_hat_out, s_out = self.spike_out(y_out, step_mode)
        return y_out, y_hat_out, s_out

cuda


## Training in multi-step mode
With the network ready, we can continue with training. We need to make two important changes:
1. Instead of `nn.CrossEntropyLoss`, we use `MultiStepCrossEntropyLoss`. This can take output from many timesteps and then compute the loss per timestep, just like for single-step mode.
2. We do not have a for loop over timesteps. Instead we pass the whole data tensor to the model at once.

Furthermore, for `SigmaDeltaSpikeFunction` it possible to set a graph mode. This is only used for multi-step mode, and determines how the spiking function is accelerated. We recommend `1step` if possible (this uses CUDA graphs), and `none` if short on memory.

In [2]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaNet(input_shape, n_outputs, n_buckets, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = MultiStepCrossEntropyLoss()
SigmaDeltaSpikeFunction.graph_mode = "1step"

We first show the *stateless* example. This means that the model will not preserve a state in between forward calls at all.

In [3]:
step_mode = StepMode.MULTI_STATELESS
model.train()
n_epochs = 3

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        y, y_hat, s = model(data, step_mode=step_mode)
        loss = loss_fn(y, labels)
        loss.backward()

        optimizer.step()
        pbar.set_description(f"Loss: {loss.item():.4f}")

Loss: 1.3687: 100%|██████████| 255/255 [00:20<00:00, 12.48it/s]


An another mode is the *stateful* mode. Here, the model will remember it's last state. This is very useful when not passing on all timesteps at once, but instead splitting it up into blocks. You would do such a thing if you run out of memory with the full data pass. We provide a helper function `multi_split_step_forward` for this, as it is a bit more complex to use than the *stateless* mode.

If you get an error starting with "T has changed from ...", this means you need to reconstruct the model. This is a safeguard against bugs where the CUDA graph is being reconstructed over and over again (very expensive), which is necessary if the number of timestep, batch size, or any shape changes. It is recommended to stick with a fixed number of timesteps for all samples, and also a fixed `T_per_split`. If this is not possible, it is better to switch away from CUDA graphs to graph_mode `none`. We set `SigmaDeltaSpikeFunction.allow_T_change_on_graph` to `True` here to override the safeguard. This is not recommended except for testing.

In [4]:
step_mode = StepMode.MULTI_STATEFUL
SigmaDeltaSpikeFunction.allow_T_change_on_graph = True
model.train()
n_epochs = 1

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        y, y_hat, s, total_loss = multi_split_step_forward(data, labels, model, loss_fn, T_per_split=10)
        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 1.5887: 100%|██████████| 255/255 [00:23<00:00, 10.89it/s]


# Testing
Works the same as training.

In [4]:
step_mode = StepMode.MULTI_STATELESS 
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        y, y_hat, s = model(data, step_mode=step_mode)
        predictions = s.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 18/18 [00:03<00:00,  5.66it/s]

Test Accuracy: 71.38%
